In [1]:
!pip install opencv-python numpy tensorflow keras imutils


  Using cached typing_extensions-4.5.0-py3-none-any.whl.metadata (8.5 kB)
Using cached typing_extensions-4.5.0-py3-none-any.whl (27 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.13.0
    Uninstalling typing_extensions-4.13.0:
      Successfully uninstalled typing_extensions-4.13.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 9.0.2 requires typing_extensions>=4.6; python_version < "3.12", but you have typing-extensions 4.5.0 which is incompatible.
torch 2.5.1 requires typing-extensions>=4.8.0, but you have typing-extensions 4.5.0 which is incompatible.

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
print(os.listdir())


['.bash_history', '.cache', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.matplotlib', '.vscode', '3D Objects', 'ANN.ipynb', 'AppData', 'Application Data', 'Array.ipynb', 'BostonData.ipynb', 'CancerDetails.ipynb', 'cifar-10-batches-py', 'cifar-10-python.tar.gz', 'CNN.ipynb', 'Contacts', 'Cookies', 'CrossDevice', 'dataset', 'DiabetesPatients.ipynb', 'Documents', 'Downloads', 'Employee.ipynb', 'Favorites', 'GoogleStockPrice.ipynb', 'ImgToPixel.ipynb', 'IntelGraphicsProfiles', 'Links', 'Local Settings', 'Mask_Dectector.ipynb', 'mask_detection_model.h5', 'mask_detection_model.keras', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{4c5c179a-e477-11ef-a58c-93625ef0ac09}.TM.blf', 'NTUSER.DAT{4c5c179a-e477-11ef-a58c-93625ef0ac09}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{4c5c179a-e477-11ef-a58c-93625ef0ac09}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'obj_detection.py', 'OneDrive', 'PrintHood', 'Re

In [4]:
print(os.listdir("C:/Users/Yash Sud/dataset/"))


['With Mask', 'Without Mask']


In [16]:
!pip install tensorflow opencv-python matplotlib numpy




[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
print("Train directory exists:", os.path.exists("dataset/train"))
print("Validation directory exists:", os.path.exists("dataset/val"))


Train directory exists: False
Validation directory exists: False


In [19]:
import os
print("Train directory exists:", os.path.exists("dataset/train"))
print("Validation directory exists:", os.path.exists("dataset/val"))


Train directory exists: True
Validation directory exists: True


In [20]:
train_dir = r"C:\Users\Yash Sud\Name\path\to\dataset\train"
val_dir = r"C:\Users\Yash Sud\path\to\dataset\val"


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# Define dataset directories
train_dir = "dataset/train"
val_dir = "dataset/val"

# Image preprocessing
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(train_dir, target_size=(224, 224), batch_size=32, class_mode='binary')
val_data = datagen.flow_from_directory(val_dir, target_size=(224, 224), batch_size=32, class_mode='binary')

# Load pre-trained MobileNetV2 model
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # Binary classification

model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(train_data, validation_data=val_data, epochs=10)

# Save the trained model
model.save("mask_detector_model.h5")

# Plot accuracy and loss
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.show()



In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load OpenCV's pre-trained face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Load trained model
model = load_model("mask_detector_model.h5")

# Start webcam capture
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (224, 224))
        face = np.expand_dims(face, axis=0) / 255.0  # Normalize

        # Make prediction
        prediction = model.predict(face)[0]
        label = "Mask" if prediction[0] > 0.5 else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Face Mask Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
